# MLTA — Colab session setup

Run this at the start of **every** session.

**One-time setup (each team member):**
1. GitHub → Settings → Developer settings → Fine-grained tokens → access to `wu1da/mlta-llm-judge-rag` only, permission *Contents: Read and write*.
2. In Colab, open the **Secrets** panel (key icon on the left), add `GH_TOKEN` = your token, and enable *Notebook access*.
3. Fill in your name / email / branch in the cell below.

Runtime: **CPU** for coding, parsing, BM25. Switch to **T4 GPU** only for embeddings and LLM generation.

**Bilingual regulatory RAG baseline**
Run the cells in order in Colab. Upload the project's ZIP when prompted. This first experiment retrieves articles from the original Official Journal editions. Recitals and annexes are outside its scope. The benchmark and RAGTruth stay separate from the retrieval index.

In [ ]:
%pip -q install lxml 'transformers>=4.48,<5' accelerate

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from google.colab import files

archive = Path('/content/MLTA-project-data.zip')
if not archive.exists():
    uploaded = files.upload()
    archive, = [Path(name) for name in uploaded if name.lower().endswith('.zip')]


Mounted at /content/drive


In [ ]:
from collections import Counter
from lxml import html
import math
import re

acts = {'ai-act': ('2024/1689', 113), 'gdpr': ('2016/679', 99)}
articles = []
with ZipFile(archive) as bundle:
    for act, (regulation, expected) in acts.items():
        for lang in ('en', 'fr'):
            path = f'project-data/eur-lex/{act}-{lang}-oj.html'
            root = html.fromstring(bundle.read(path))
            nodes = [n for n in root.xpath('//div[@id]') if re.fullmatch(r'art_\d+', n.get('id'))]
            assert {n.get('id') for n in nodes} == {f'art_{i}' for i in range(1, expected+1)}
            for node in nodes:
                number = int(node.get('id')[4:])
                title = ' '.join(node.xpath('.//*[contains(@class, "sti-art")]//text()')).strip()
                body = ' '.join(' '.join(node.itertext()).split())
                language = {'en':'eng', 'fr':'fra'}[lang]
                url = f'https://eur-lex.europa.eu/eli/reg/{regulation}/oj/{language}#art_{number}'
                articles.append(dict(act=act, lang=lang, number=number, title=title, body=body, url=url))
print(Counter((a['act'], a['lang']) for a in articles))

In [ ]:
# Build one BM25 index per language. Preserve article links on every passage.
from collections import defaultdict
indexes = {}
for lang in ('en', 'fr'):
    passages = []
    df = Counter()
    for art in (a for a in articles if a['lang'] == lang):
        words = art['body'].split()
        for start in range(0, len(words), 110):
            text = ' '.join(words[start:start+160])
            counts = Counter(re.findall(r'\w+', (art['title']+' '+text).casefold()))
            passages.append((art, text, counts, sum(counts.values())))
            df.update(counts.keys())
            if start + 160 >= len(words):
                break
    indexes[lang] = passages, df, sum(p[3] for p in passages)/len(passages)

def retrieve(question, lang, k=3):
    passages, df, avg = indexes[lang]
    terms = set(re.findall(r'\w+', question.casefold()))
    hits = []
    for art, text, counts, length in passages:
        score = 0
        for term in terms.intersection(counts):
            idf = math.log1p((len(passages)-df[term]+0.5)/(df[term]+0.5))
            tf = counts[term]
            score += idf * tf * 2.2/(tf + 1.2*(0.25 + 0.75*length/avg))
        if score:
            hits.append((score, art, text))
    return sorted(hits, key=lambda h: -h[0])[:k]

for lang, query in [('en','right to data portability'), ('fr','droit à la portabilité des données')]:
    first = retrieve(query, lang)[0]
    print(lang, first[1]['act'], first[1]['number'], first[1]['title'], first[1]['url'])


**Generate an answer from the retrieved evidence**
A Colab GPU is recommended for this cell. The first run downloads Qwen2.5-1.5B-Instruct. Compare the answer with the returned passages to check whether each citation supports the claim.

In [ ]:
from transformers import pipeline
generator = pipeline('text-generation', model='Qwen/Qwen2.5-1.5B-Instruct',
                     device_map='auto', torch_dtype='auto')

def answer(question, lang='en'):
    hits = retrieve(question, lang)
    abstention = {'en':'Insufficient evidence.', 'fr':'Éléments insuffisants.'}[lang]
    if not hits:
        return abstention, []
    evidence = []
    for i, (_, art, text) in enumerate(hits, 1):
        evidence.append(f"[{i}] {art['act']} Article {art['number']}: {text}")
    instructions = (f'Answer in {lang} using only the supplied passages. Cite each claim with '
                    f'its passage number in brackets. If the passages do not answer the '
                    f'question, reply exactly: {abstention}')
    messages = [{'role':'system','content':instructions},
                {'role':'user','content':question + chr(10)*2 + (chr(10)*2).join(evidence)}]
    result = generator(messages, max_new_tokens=180, do_sample=False, return_full_text=False)
    return result[0]['generated_text'], [(art['url'], text) for _, art, text in hits]

response, sources = answer('What is the GDPR right to data portability?')
print(response)
for i, (url, passage) in enumerate(sources, 1):
    print(f'[{i}]', url, passage)
